# iTantra Tamil/Telugu 10-hour trial â€” Gate B (NOT RUN)
First run `kathbath_gate_a.ipynb` and return its audit/leakage report. This notebook stays blocked until that report is reviewed. Tamil runs first; Telugu training follows only after Tamil's trial is reviewed. Do not scale beyond the 10-hour trial without approval.
Use a Linux cloud GPU with enough VRAM/disk, the accepted gated AI4Bharat checkpoint, and *separate* private storage for raw audio/checkpoints. This notebook is a guarded starting point, not a measured improvement. Never train on the 30 validation clips, any FLEURS validation/test clip, or matching normalized sentences. Keep the original app packs until all held-out and phone checks pass.

Transfer this repository's current tools plus ignored protected indexes/manifests/test WAV caches to the cloud. Put checkpoints outside the repository. The repo's leakage check must pass before each run. For speaker-collected audio, retain explicit consent records outside Git.


In [ ]:
import os, sys, json, subprocess, pathlib, platform
GATE_A_REVIEWED = False  # change only after the user's Gate A approval
assert GATE_A_REVIEWED, 'STOP: return the Gate A audit and leakage report before training'
REPO = pathlib.Path(os.environ.get('ITANTRA_REPO', '/kaggle/working/itantra')).resolve()
LANG = 'ta'  # Telugu only after the Tamil trial is reviewed; bn/gu out of scope
assert LANG in ('ta', 'te')
SEEDS = [17, 29]
TRAIN = REPO / f'tools/stt_training/manifests/{LANG}-train.jsonl'
DEV = REPO / f'tools/stt_training/manifests/{LANG}-dev.jsonl'
PROTECTED = REPO / f'tools/stt_training/{LANG}-combined-protected.json'
TEST30 = REPO / f'tools/stt_models/five-language-validation/manifest-{LANG}-30.json'
TEST100 = REPO / f'tools/stt_models/five-language-test/manifest-{LANG}-100.json'
OUT = pathlib.Path(os.environ.get('ITANTRA_TRAIN_OUT', '/kaggle/working/itantra-training')).resolve()
assert all(p.exists() for p in (TRAIN, DEV, PROTECTED, TEST30, TEST100)), 'Transfer all manifests and full protected index first'
subprocess.run([sys.executable, str(REPO/'tools/check_speech_leakage.py'), 'workspace-check'], check=True)
print('GPU:', subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip())
print('Git:', subprocess.run(['git','rev-parse','HEAD'], cwd=REPO, capture_output=True, text=True).stdout.strip())


## Install training runtime, then restart the kernel
The checkpoint cards require the [AI4Bharat NeMo `nemo-v2` fork](https://github.com/AI4Bharat/NeMo/tree/nemo-v2). On a fresh Linux GPU runtime run the commands below, record the resolved commit and package versions, then restart the kernel. Installation has **not** been tested on this Windows host.
```bash
git clone --branch nemo-v2 https://github.com/AI4Bharat/NeMo.git /kaggle/working/NeMo
cd /kaggle/working/NeMo && git rev-parse HEAD && bash reinstall.sh
python -m pip install onnx onnxruntime sherpa-onnx soundfile librosa
```
Accept access separately for each `ai4bharat/indicconformer_stt_<lang>_hybrid_ctc_rnnt_large` checkpoint. Put the Hugging Face read token in a cloud secret named `HF_TOKEN`; never save it in this notebook or logs.


In [ ]:
import torch, nemo.collections.asr as nemo_asr
from huggingface_hub import hf_hub_download
token = os.environ.get('HF_TOKEN')
assert token, 'Provide the gated-checkpoint token through a cloud secret/environment variable'
assert torch.cuda.is_available(), 'Training requires a usable CUDA GPU'
model_id = f'ai4bharat/indicconformer_stt_{LANG}_hybrid_ctc_rnnt_large'
revisions = {'ta':'8c31aa8d04964b8fc87e4eaaee7916f7d2c024da', 'te':'4133836fef3b0687fdffaa32512f3496657c0b48'}
checkpoint = hf_hub_download(repo_id=model_id, filename=f'indicconformer_stt_{LANG}_hybrid_rnnt_large.nemo', revision=revisions[LANG], token=token)
model = nemo_asr.models.ASRModel.restore_from(checkpoint)
print('Restored:', type(model).__name__, 'CUDA device:', torch.cuda.get_device_name(0))


## Pre-training baseline gate
Decode the same 30 untouched clips with **both** CTC and RNNT before training. Compare CTC with the app's INT8 baseline (ta 21.8%, te 22.7%). A different FP32/PyTorch path can differ slightly; investigate any material gap before spending GPU hours. These clips remain evaluation-only.


In [ ]:
sys.path.insert(0, str(REPO/'tools'))
from validate_stt import normalize, distance
def unwrap(result):
    while isinstance(result, list): result = result[0]
    return result if isinstance(result, str) else result.text
records = json.loads(TEST30.read_text(encoding='utf-8'))
baseline = {}
model.eval().cuda()
for head in ('ctc', 'rnnt'):
    model.cur_decoder = head
    err = words = 0
    for row in records:
        result = model.transcribe([str(REPO/row['file'])], batch_size=1, language_id=LANG)
        ref = normalize(row['reference']).split()
        hyp = normalize(unwrap(result)).split()
        err += distance(ref, hyp); words += len(ref)
    baseline[head] = err / words
print('PyTorch held-out 30 WER (CTC/RNNT):', baseline)
expected = {'ta': .218274, 'te': .226981}[LANG]
assert abs(baseline['ctc'] - expected) <= .03, 'Explain baseline gap before training; normalization, export mask or checkpoint may differ'
OUT.mkdir(parents=True, exist_ok=True)
(OUT/f'{LANG}-pytorch-baseline.json').write_text(json.dumps(baseline, indent=2))


## Fine-tuning commands (inspect NeMo version before running)
Use the fork's `examples/asr/speech_to_text_finetune.py` and its actual config. Start with a low learning rate, two seeds and a separate frozen-encoder pilot. Enable SpecAugment and add only licensed noise/reverb/codec augmentation sources, recorded in the data audit. Early-stop on **dev** WER and keep the best checkpoint. Never use either held-out test set for early stopping or hyperparameter selection. Budget planning only: reserve roughly 4â€“12 GPU-hours *per run* on a suitable cloud GPU, then replace that estimate with measured time. The seed labels below do not yet seed the upstream trainer. Gate B still needs runtime validation of seeds, mixed precision, speed augmentation and dev early stopping. The commands below are templates because NeMo is absent on this host; check `--help` and resolved Hydra config on the target runtime.


In [ ]:
NEMO_ROOT = pathlib.Path('/kaggle/working/NeMo')
base_checkpoint = OUT/f'{LANG}-base.nemo'
model.cpu().save_to(str(base_checkpoint))
for seed in SEEDS:
    cmd = [sys.executable, str(NEMO_ROOT/'examples/asr/speech_to_text_finetune.py'),
           f'--config-path={NEMO_ROOT / "examples/asr/conf/asr_finetune"}', '--config-name=speech_to_text_finetune',
           f'init_from_nemo_model={base_checkpoint}',
           f'model.train_ds.manifest_filepath={TRAIN}', f'model.validation_ds.manifest_filepath={DEV}',
           'trainer.accelerator=gpu', 'trainer.devices=1', 'trainer.max_epochs=30',
           'model.optim.lr=2e-5', 'model.optim.sched.warmup_steps=100', f'exp_manager.exp_dir={OUT/LANG/str(seed)}']
    print('Review resolved config, augmentation, precision, early stopping, freeze pilot and checkpoint path before running:')
    print(' '.join(map(str, cmd)))
# After review, run each command on the cloud runtime and save logs/configs/checkpoints outside Git.


## Export and independent evaluation
Once training finishes, run `tools/export_finetuned_indic_ctc.py` on the **best** checkpoint using a TRAIN/DEV trace clip. Its MatMul-only INT8 export follows the current publisher's masked-CTC format and fails if sherpa cannot load/decode it. Compare PyTorch, FP32 and INT8 on the 30 clips, 100 distinct FLEURS test sentences and the protected Kathbath noisy test_unknown set (phone testing remains separate), each separately. Run each `evaluate_stt_preprocessing.py` command in a fresh process. Check model bytes, mean/p95 decode, RTF, peak RAM, script rate and seed variance. Do not promote to the app unless all three held-out sets improve clearly and phone testing passes.

Example: `python tools/export_finetuned_indic_ctc.py --language ta --checkpoint /outside-repo/best.nemo --trace-wav /outside-repo/train.wav --output-dir /outside-repo/ta-candidate`

Example: `python tools/evaluate_stt_preprocessing.py --language ta --method baseline --set 100 --model-dir /outside-repo/ta-candidate --label ta-seed17-test`
